# Inspect and prepare COMEX gold futures data

Use this notebook to download and inspect Yahoo Finance daily COMEX gold futures (`GC=F`) before model work. This is a continuous futures series, not spot gold or a single fixed-expiry contract; returns may reflect contract rolls. The default target is forward five-observation realized variance (sum of squared close-to-close log returns); realized volatility is also available.

**Important:** target values use future observations by design and are labels, not predictors available at the forecast origin. This notebook does not fit or evaluate a forecasting model. `GC=F` is futures data; do not describe it as spot. Record source, instrument, currency, units, and observation dates; do not share restricted data.

Install the project and notebook dependencies first using the README instructions.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from gold_vol_pipeline import (
    DEFAULT_HORIZON,
    DEFAULT_TARGET,
    download_futures_prices,
    load_gold_prices_csv,
    prepare_dataset,
)

## 1. Set the inputs

Change these values and rerun the notebook to inspect a different sample, forecast horizon, or target measure. The horizon counts subsequent price observations, not calendar days. Yahoo Finance is the default data source; you can alternatively select a local CSV.

In [ ]:
START_DATE = "2015-01-01"
END_DATE = pd.Timestamp.now().normalize().strftime("%Y-%m-%d")  # exclusive; omit today's partial bar
DATA_SOURCE = "yahoo"  # "yahoo" or "csv"
CSV_PATH = Path("data/raw/gold_futures.csv")
CSV_DATE_COLUMN = "date"
CSV_CLOSE_COLUMN = "close"
CSV_SOURCE_LABEL = "Enter the actual data provider"
CSV_INSTRUMENT_LABEL = "Enter the actual futures contract or series"
CSV_CURRENCY = "USD"
CSV_PRICE_UNIT = "USD per troy ounce"
HORIZON = DEFAULT_HORIZON
TARGET = DEFAULT_TARGET  # "realized_variance" or "realized_volatility"
SAVE_OUTPUTS = True

print(f"Data source: {'Yahoo Finance (GC=F)' if DATA_SOURCE == 'yahoo' else CSV_SOURCE_LABEL}")
if DATA_SOURCE == "yahoo":
    print(f"Requested range: [{START_DATE}, {END_DATE})")
else:
    print(f"Instrument: {CSV_INSTRUMENT_LABEL}")
print(f"Target: {HORIZON}-observation forward {TARGET}")

## 2. Download or load and inspect the raw prices

Review the returned sample and date coverage before proceeding. The pipeline does not fill missing dates or interpolate prices.

In [ ]:
if DATA_SOURCE == "yahoo":
    prices = download_futures_prices(START_DATE, END_DATE)
elif DATA_SOURCE == "csv":
    prices = load_gold_prices_csv(
        CSV_PATH,
        date_column=CSV_DATE_COLUMN,
        close_column=CSV_CLOSE_COLUMN,
        source=CSV_SOURCE_LABEL,
        instrument=CSV_INSTRUMENT_LABEL,
        currency=CSV_CURRENCY,
        price_unit=CSV_PRICE_UNIT,
    )
else:
    raise ValueError("DATA_SOURCE must be 'yahoo' or 'csv'.")
prices.head(10)

In [ ]:
assert not prices.empty, "No prices were returned. Check the date range and data source."
assert prices["date"].notna().all(), "Raw data contains missing dates."
assert prices["close"].notna().all(), "Raw data contains missing closes."
assert np.isfinite(prices["close"]).all(), "Raw data contains non-finite closes."
assert (prices["close"] > 0).all(), "Raw data contains non-positive closes."
assert not prices["date"].duplicated().any(), "Raw data contains duplicate dates."

gaps = prices["date"].sort_values().diff().dt.days.dropna()
print(f"Rows: {len(prices):,}")
print(f"Date range: {prices['date'].min().date()} to {prices['date'].max().date()}")
print(f"Close range: {prices['close'].min():,.2f} to {prices['close'].max():,.2f} USD/oz")
print("Calendar-day gaps between observations (including weekends/holidays):")
display(gaps.value_counts().sort_index().rename_axis("days").to_frame("count").head(12))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(prices["date"], prices["close"], linewidth=1)
ax.set(title="COMEX gold futures closing price", xlabel="Date", ylabel=CSV_PRICE_UNIT)
ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()

## 3. Build the prepared dataset

Each row is a forecast origin after that day's close. Its target window starts at the next observed price date and ends after `HORIZON` observed returns. The first target return therefore is not the return ending on the forecast-origin date.

In [ ]:
dataset = prepare_dataset(prices, horizon=HORIZON, target=TARGET)
display(dataset.head(10))
display(dataset.tail(5))

## 4. Check target alignment and values

The checks below independently recalculate a few targets from raw prices. They help catch accidental changes to the target-window convention. A successful check does not establish that the source data or target definition is appropriate for every research question.

In [ ]:
price_dates = pd.to_datetime(prices["date"]).dt.normalize().reset_index(drop=True)
price_closes = prices["close"].astype(float).reset_index(drop=True)
price_log_returns = np.log(price_closes).diff()
target_column = TARGET
checks = min(10, len(dataset))

assert dataset[target_column].notna().all(), "Prepared target contains missing values."
assert np.isfinite(dataset[target_column]).all(), "Prepared target contains non-finite values."
assert (dataset[target_column] >= 0).all(), "Realized variance/volatility should be non-negative."

for row in dataset.head(checks).itertuples(index=False):
    origin = price_dates[price_dates == row.date].index[0]
    future_returns = price_log_returns.iloc[origin + 1 : origin + HORIZON + 1]
    assert len(future_returns) == HORIZON
    expected_variance = float((future_returns**2).sum())
    expected = np.sqrt(expected_variance) if TARGET == "realized_volatility" else expected_variance
    assert np.isclose(getattr(row, target_column), expected, rtol=1e-10, atol=1e-14)
    assert row.target_start_date == price_dates.iloc[origin + 1]
    assert row.target_end_date == price_dates.iloc[origin + HORIZON]

print(f"Passed target and date-window checks for {checks} forecast origins.")
print(f"Prepared rows: {len(dataset):,}; dropped incomplete trailing targets: {len(prices) - len(dataset):,}")

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
axes[0].plot(prices["date"], prices["close"], linewidth=0.8)
axes[0].set_ylabel("USD/oz")
axes[0].set_title("Price, daily log returns, and forward target")
axes[1].plot(dataset["date"], dataset["log_return"], linewidth=0.6)
axes[1].set_ylabel("Log return")
axes[2].plot(dataset["date"], dataset[target_column], linewidth=0.8)
axes[2].set_ylabel(TARGET.replace("_", " ").title())
axes[2].set_xlabel("Forecast-origin date")
for ax in axes:
    ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()

## 5. Save this run (optional)

Raw and processed data are git-ignored. The small JSON manifest records the requested date range, options, provider, and download time so the saved files can be traced to this run. Preserve the raw CSV used for any model comparison.

In [ ]:
if SAVE_OUTPUTS:
    run_id = f"{prices['date'].min():%Y%m%d}_to_{prices['date'].max():%Y%m%d}_h{HORIZON}_{TARGET}"
    raw_path = Path("data/raw") / f"gold_futures_{run_id}.csv"
    processed_path = Path("data/processed") / f"gold_futures_{run_id}.csv"
    manifest_path = Path("data/processed") / f"gold_futures_{run_id}.json"
    raw_path.parent.mkdir(parents=True, exist_ok=True)
    processed_path.parent.mkdir(parents=True, exist_ok=True)
    prices.to_csv(raw_path, index=False)
    dataset.to_csv(processed_path, index=False)
    manifest = {
        "provider": prices.attrs.get("source", "CSV source not set"),
        "instrument": prices.attrs.get("instrument", "Instrument not set"),
        "currency": prices.attrs.get("currency", CSV_CURRENCY),
        "price_unit": prices.attrs.get("price_unit", CSV_PRICE_UNIT),
        "observation_start_date": prices["date"].min().strftime("%Y-%m-%d"),
        "observation_end_date": prices["date"].max().strftime("%Y-%m-%d"),
        "raw_file": str(raw_path),
        "horizon_observations": HORIZON,
        "target": TARGET,
        "return_definition": "close-to-close log return",
        "target_definition": "sum of next horizon squared log returns"
        if TARGET == "realized_variance"
        else "square root of sum of next horizon squared log returns",
        "pipeline_run_at_utc": pd.Timestamp.now(tz="UTC").isoformat(),
        "prepared_file": str(processed_path),
    }
    manifest_path.write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
    print(f"Saved raw prices: {raw_path}")
    print(f"Saved prepared data: {processed_path}")
    print(f"Saved run manifest: {manifest_path}")
else:
    print("SAVE_OUTPUTS is False; no files were written.")